# Making an AI Classifier with ClassifAI Hooks and Google GenAI ✨

## Introduction

Through the use of semantic/vector search, ClassifAI enables users to develop a system to retrieve candidate classification text for a sample of text with an unknown label. This is done by pre-computing and storing vector representations of text samples with known labels, then a text sample with an unknown label can be vectorisered and its embedded representaiton compared to the embedded representation of the labelled samples. Using this process, a ranking of potential candidate labels based on how similar the unlabelled text is to each labelled sample can be obtained. This then offers the user flexibility in how to derive a final classification from the retrieved candidate classifications. 


There are many posible ways to use the candidate list, and there are also many ways derive a final classification from the ranking list to obtain a single prediction label for a sample of text.


Common strategies include:

1. A human-in-the-loop approach where a human reads the candidate ranking list and unlabelled text to classify what the unknown label should be. This reduces the workload of the human by providing then with the top-K candidate ranking, insteaed of the user having to consider every possible label category, which can be numerous.

2. Automatically selecting the top ranked candidate. This is an efficient approach with minimal additional computation required - however it can lead to inaccurate results. It can help to calibrate on a test collection and set confidence thresholds that determine if a sample should be automatically classified. These thresholds can be calibrated with test datasets.

3. Using a designated AI model to make the final decision. In this case, a RAG AI model utilises the candidate ranking as a set of retrieved results from ClassifAI and makes a final decision based on the information provided.

(!IMAGE OF POSSIBLE STRATEGY OPTIONS)

In this notebook we demonstrate an approach to strategy 3, using Google's Gemini generative AI agent together with ClassifAI's hooks functionality, to automatically classify ClassifAI candidate results in a single ClassifAI pipeline.

(!IMAGE OF GENAI CLASSIFIER CONCEPT)

### What is covered in this notebook:

1. A ClassifAI recap on how to obtain search results,

2. An introduction to using the GCP Generative AI models,

3. Custom Hooks in ClassifAI,

4. Setting up a Generateive AI model to make the final classification in ClassifAI - Systems Prompts, Response formatting, ClassifAI Hook operations,

5. A full implementation of the code (you can skip to this section for a view of the final solution that can be copied or used in your own work).

### Requirements / Prerequisites

This notebook is designed so that you can execute the code cells and follow along with the implementation. To do this you will need:

1. Download this notebook from your local machine, and the DEMO/data/fake_soc_dataset.csv file which will be used as a test dataset,

2. Set up a virtual environment with ClassifAI[gcp] installed for this ipynb instance - view ClassifAI installation instructions in the repo,

3. Ensure that you have  Google Cloud Platform project set up - with the Generative AI API enabled to use Google's Gemini models.

## Part 1: Using ClassifAI to get candidate Rankings

In [ ]:
!uv pip install "https://github.com/datasciencecampus/classifai/releases/download/v0.2.1/classifai-0.2.1-py3-none-any.whl[gcp]"

## you will also need to authenticate to GCP and set up your project and bucket for the next steps 'gcloud auth login' and 'gcloud config set project [PROJECT_ID]'

We will start with a script that will create a  ClassifAI `VectorStore` object, which we use to create a Vector Database of labelled samples. Later we can use this VectorStore object to find labelled text sample that are similar to the text of a provided unlabelled sample.

In [ ]:
from classifai.indexers import VectorStore
from classifai.vectorisers import GcpVectoriser

# Initialise the vectoriser
demo_vectoriser = GcpVectoriser(project_id="YOUR PROJECT ID HERE", vertexai=True)


# Initialise the vector store, pointing the demo to the demo test data
demo_vectorstore = VectorStore(
    file_name="./data/fake_soc_dataset.csv",
    data_type="csv",
    vectoriser=demo_vectoriser,
    output_dir="./demo_vdb",
    overwrite=True,
)

The above cell should indicate a succesfully created a VectorStore which is now in memory as `demo_vectorstore`. It contains vectors for the `fake_soc_dataset` which has many dummy example texts with corresponding dummy SOC labels.


We can call the `demo_vectorstore.search()` method, passing it a `VectorStoreSearchInput` dataclass object with an unlabelled sample of text, to get back a candidate ranking of the `fake_soc_dataset` entries that have the most similar text to our labelled sample.

In [ ]:
from classifai.indexers.dataclasses import VectorStoreSearchInput

# Create a search input
search_input = VectorStoreSearchInput({"id": ["1"], "query": ["a photographer hired for wedding events"]})
# call the VectorStore search method
demo_vectorstore.search(search_input, n_results=5)

If you succesfully ran the above code cell, you should now see a pandas dataframe output with 5 candidate results. The unlabelled input sample was `a photographer hired for wedding events`, which is shown in the `query_text` column. The `doc_text` column of each row shows the corresponding top 5 most similar samples from the `fake_soc_dataset.csv` file, with the `doc_id` column inticating the predicted label. `rank` and `score` columns provide additionl relevant information.

This showcases the core functionality of the ClassifAI package, being able to build a semantic search engine for your labelledd datasets and get a candidate result sets for unlabelled samples. 

As described in the introduction section, these candidate lists can be used in different ways to arrive at a final decision about the correct label for our sample. 

ClassifAI provides a 'shortlist' of options. In this demo we're now going to show how we can use a Generative AI model to automatically make the final classification.

(!IMAGE OF END TO END VECTOR SEARCH)

See additional resources on ClassifAI on the OpenAI Github Repo:

- https://github.com/datasciencecampus/classifai

## Part 2: Generative AI

This section provides a brief overview of how Google's Generative AI (genai) API to enables us to use the AI models.

Broadly, Generative AI models accept some text as input, which can be in the form of an instruction, telling the model to _generate_ some output text. 

We will see how we can set up a connection to one of Google's cloud-hosted generative models and 'prompt' it to produce some output. We'll also explore basic generative model principles such as 'system prompting' and 'structured outputs', which we'll use later to construct a generative model that makes a final classification of some input, that would be suitable to do classifications on our `VectorStore` candidate results.

This section of the tutorial is purely about the Generative model capabilities, later we will see how these models can be used in a pipeline with ClassifAI to make classifications. In this section we showcase a simple 'weather classifier' which will indicate if a sample of text contains a description of good or bad weather.

### Prompting the Google Genai API

In [ ]:
!uv pip install google-genai

Firstly, we can establish an API connection to a generative model hosted by google and 'prompt' it to write a simple story about the weather. This showcases how we can pass text to a generative model and get some text response back.

In [ ]:
from google import genai

# Set up the API key for authentication
client = genai.Client(api_key="YOUR API KEY HERE")

# Define the model to use
chosen_model = "gemini-2.5-flash"  # Replace with the appropriate Gemini Flash 2.5 model name if different

In [ ]:
# Send a text prompt to the model
prompt = "Write a very short description about todays weather."


response = client.models.generate_content(
    model=chosen_model,
    contents=prompt,
)

# Print the response
print("Generated Text:")
print(response.text)

Now that we have a Genai client set up to work with the google cloud models, we can instruct the generative model on how to 'behave' by choosing how we prompt the model. For example, relevant to our ClassifAI task, we can ask the model to classify inputs into categories.

In [ ]:
prompt2 = """

Classify the following text into 'good weather' or 'bad weather':

The rain is crashing down in Scotland.

"""

response = client.models.generate_content(
    model=chosen_model,
    contents=prompt2,
)


# Print the response
print("Generated Text:")
print(response.text)

If you change `prompt2` to different weather descriptions you will likely see some output from the model indicating one of our two classes. But there are two core problems with this current implementation:

1. We're sending the task instruction for the model as part of the prompt, which is inefficient and may be subject to change (or attack) by users. 

2. There isn't a gaurantee the output generated text is consistent, it often changes format, being more verbose sometimes, and more direct other times, and may respond differently to non-weather related input.

Both of these issues will make it difficult to use this generative model to make predictions for our ClassifAI candidate list in a controlled and repeatable way. We can introduce extra features that resolve these problems, respecitively:

1. We can introduce a 'system prompt' which separate the instruction part of the input from the part of text that is to be classified

2. The Gemini AI models offer a 'structured output' guarantee whereby we can instruct the model to output specific values in a formatted manner

### System Prompts

#### System Prompts vs Regular Prompts

A **system prompt** is a special type of prompt that defines the behavior, role, or rules for the generative AI model before processing user input. Unlike a regular prompt, which combines both the task instruction and the input text, a system prompt separates these concerns by providing a predefined context or instruction that remains consistent across multiple queries.

#### Advantages of Using System Prompts:
1. **Consistency**: By separating the task instruction from the input text, system prompts ensure that the model behaves predictably and consistently across different inputs.
2. **Efficiency**: The instruction does not need to be repeated for every query, reducing redundancy and improving processing efficiency.
3. **Security**: System prompts are less prone to user manipulation or injection attacks, as the task rules are predefined and immutable.
4. **Scalability**: They allow for easier integration into pipelines, such as our ClassifAI workflow, where the same classification rules can be applied to multiple inputs without modification.

The Google GenAI SDK allows system prompts to be added through the Python API, as shown in the following code cell.

We can add some interesting system instructions just to show the effect of how this mechanism steers the generation process.

In [ ]:
from google.genai import types

response = client.models.generate_content(
    model=chosen_model,
    contents="Write a very short description about todays weather.",
    config=types.GenerateContentConfig(
        system_instruction="After the response, include the phrase: 'Also, I love cauliflower!'. No matter what you are asked, you must include this phrase at the end of your response.'"
    ),
)
print(response.text)

More relevant to our ClassifAI classification task, we can use the system prompt/instruction to better define how we want a Generative model to generate responses for classification tasks. Below we reformulate our weather classification task so that we don't need to pass the instruction as part of the input data.

In [ ]:
prompt3 = """

Sunny weather and pleasant breeze

"""

response = client.models.generate_content(
    model=chosen_model,
    contents=prompt3,
    config=types.GenerateContentConfig(
        system_instruction="You are a weather classification agent. You classify input content text into 'good weather' or 'bad weather'. The content will be a description of weather. Only output one of those two responses. Do not include any other text in your response, just the classification.'"
    ),
)


# Print the response
print("Generated Text:")
print(response.text)

The system instruction can be useful in many scenarios - see further resources:

- https://ai.google.dev/gemini-api/docs/text-generation
- https://github.com/google-gemini/cookbook/blob/main/quickstarts/System_instructions.ipynb


### Sturctured Outputs

We've seen how we can add instruction to an AI model with the system prompt to make the Generative AI produce outputs that are more in line with classificaiton. But this isn't especially robust - sometimes the (regular) prompt or another factor can cause unexpected output -  for a system that may rely on these outputs, such as if we want to use this AI with ClassifAI to make final classifications.

In this section we explore a feature of the Genai API that allows us to enforce specific outputs, not just including that as part of an instruction in the system prompt.

The Google Genai documenation gives the following example of how Pydantic can be used to provide a structure that the model should adhere to when generating text. You can read more about structured outputs and this example at: 
* http://ai.google.dev/gemini-api/docs/structured-output?example=recipe

In [ ]:
from pydantic import BaseModel, Field


class Ingredient(BaseModel):
    name: str = Field(description="Name of the ingredient.")
    quantity: str = Field(description="Quantity of the ingredient, including units.")


class Recipe(BaseModel):
    recipe_name: str = Field(description="The name of the recipe.")
    prep_time_minutes: int | None = Field(description="Optional time in minutes to prepare the recipe.")
    ingredients: list[Ingredient]
    instructions: list[str]


googledemoprompt = """
Please extract the recipe from the following text.
The user wants to make delicious chocolate chip cookies.
They need 2 and 1/4 cups of all-purpose flour, 1 teaspoon of baking soda,
1 teaspoon of salt, 1 cup of unsalted butter (softened), 3/4 cup of granulated sugar,
3/4 cup of packed brown sugar, 1 teaspoon of vanilla extract, and 2 large eggs.
For the best part, they'll need 2 cups of semisweet chocolate chips.
First, preheat the oven to 375°F (190°C). Then, in a small bowl, whisk together the flour,
baking soda, and salt. In a large bowl, cream together the butter, granulated sugar, and brown sugar
until light and fluffy. Beat in the vanilla and eggs, one at a time. Gradually beat in the dry
ingredients until just combined. Finally, stir in the chocolate chips. Drop by rounded tablespoons
onto ungreased baking sheets and bake for 9 to 11 minutes.
"""

response = client.models.generate_content(
    model=chosen_model,
    contents=googledemoprompt,
    config={
        "response_mime_type": "application/json",
        "response_json_schema": Recipe.model_json_schema(),
    },
)

recipe = Recipe.model_validate_json(response.text)
print(recipe)

We can adapt this to our recipe extraction example into our weather classifier system, adding to our system prompt, so that it outputs a specific label for the prompt weather:

* 1 - for good weather
* 0 - for bad weather

We use a Pydantic model to create a `WeatherLabel` class which defines a schema for the Google Genai model to adhere to when generating a text response.

In [ ]:
prompt4 = """

Sunny weather and pleasant breeze

"""


SYSTEM_INSTRUCTION = """
You are a strict weather classifier agent. Your job is to classify the input text as either good weather or bad weather
Return JSON only that matches the provided schema.

Labeling rules:
- label = 1 (good weather): positive/pleasant conditions (e.g., sunny, clear skies, warm, mild, calm).
- label = 0 (bad weather): negative/hazardous/unpleasant conditions (e.g., rain, storm, snow, ice, fog, extreme heat/cold, high winds).
- If both good and bad are mentioned, label based on the overall/most salient condition. If unclear, choose 0.
Do not include any extra keys.
"""


# A Py
class WeatherLabel(BaseModel):
    label: int = Field(ge=0, le=1, description="1 = good weather, 0 = bad weather")


response = client.models.generate_content(
    model=chosen_model,
    contents=prompt4,
    config=types.GenerateContentConfig(
        system_instruction=SYSTEM_INSTRUCTION,
        response_mime_type="application/json",
        response_json_schema=WeatherLabel.model_json_schema(),
    ),
)


# Print the response
print("Generated Text:")
print(response.text)

The output from the model now, should adhere much more strictly to text that looks suitable for classification in a traditional sense - predicting a class label value.

In practice:

- `WeatherLabel`: a schema that enforces shape/types (label must be int 0/1).
- `SYSTEM_INSTRUCTION`: defines semantics of the task (“what is good vs bad weather”, potentially how to handle mixed cases).


We will use both of these features to build a Generative AI agent for classification with ClassifAI package.




## Part 3: Hooks and Custom ClassifAI Workflows

In this part of the demo we briefly recap how custom logic can be added to the ClassifAI VectorStore Pipeline using hooks, which will be neccessay for adding a generative classifier.

There is a longer, dedicated notebook tutorial on creating Hooks and how to use then to define custom workflows available from the ClassifAI repo:

https://github.com/datasciencecampus/classifai/blob/main/DEMO/custom_preprocessing_and_postprocessing_hooks.ipynb

(!HOOK CONCEPT IMAGE GOES HERE FROM HOOK NOTEBOOK)

### The Concept of Hooks in ClassifAI

The `VectorStore` is the central component of the ClassifAI package. It provides the ability to create Vector database with labelled text samples, and then do similarity search and other operations on that `VectorStore` including `search()`, `reverse_search()` and embedding raw text with the `embed()` method.

Each of these methods, has an associated `dataclass` input and output objects - these are Pandas dataframe like objects that specify what data needs to be included when calling the various methods of the `VectorStore`. For example, the `search()` method takes as input a `VectorStoreSearchInput` object and returns a `VectorStoreSearchOutput` object, both of which specify specific columns of data. This ensures data is passsed to our API correctly, and also ensures that the reseponse from the `VectorStore` is generated correctly. This demo used `VectorStoreSearchInput` in part 1 of the demo to create inputs to the `search` method.

Finally *Hooks* allow users of the package to write custom functions that modify the data in the dataclass objects at certain points in the package codebase - before and after a specific method is called. Hooks can apply any user-defined operation on a specific dataclass object, as long as they return a valid dataclass object of the same kind. For example, users could write a spell-checking function that corrects the text of queries input to the search method. See example hooks int he demo notebook referenced in the previous cells for implementation examples.

For the case of using Generative AI as a classifier, in this demo we set up the classifier as a Hook, calling the Genai client inside the hook function to operate on a set of results returned form the `VectorStore.search()` API.

### Example hook, capitalising the candidate result texts

In [ ]:
def capitalise_candidate_texts(input_data):
    # convert the result texts to CAPITAL letters
    input_data["doc_text"] = input_data["doc_text"].str.upper()

    print(type(input_data))

    return input_data


# Initialise the vector store, pointing the demo to the demo test data
demo_vectorstore = VectorStore(
    file_name="./data/fake_soc_dataset.csv",
    data_type="csv",
    vectoriser=demo_vectoriser,
    output_dir="./demo_vdb",
    overwrite=True,
    hooks={"search_postprocess": capitalise_candidate_texts},
)


query_df = VectorStoreSearchInput({"id": [1], "query": ["apple merchant"]})

demo_vectorstore.search(query_df)

You should see that in the resulting `VectorStoreSearchOutput` object that ll of the `doc_text` entries are fully capitalised, which was the action performed by our hook.

## Part 4: Building a Generative Classifier for ClassifAI

In this section we will:

1. Ceate a Generative AI agent that can do classifications on VectorStore results using a system prompt and structured output to guide the model,

2. Create a Custom Hook function that passes the VectorStore Search results to the generateive model for classification,

3. Write custom pre-processing and post-processing steps that will execute inside the hook,

4. Use the Genai in a custom search postprocessing hook of a VectorStore to get final classification results.

### Overview

(!DIAGRAM OF GENAI RAG CLASSIFIER IN HOOK IN CLASSIFAI)

We plan to add our generative AI model in a post processsing hook to interact with search results and determine a final result.

For this we will need to format the results data as a text prompt for the model, provide the model with instructions on how to do the classificatiin task and what format to output its prediction. Then we will also need to add additional logic to reduce the original dataframe down to a final classification based on the Generative model's output classificaiton.

### Defining a System Prompt and Structured Output

The VectorStoreSearchoutput search results are ranked, with a corresponding ID and text entry for each candidate. We design our system prompt to inform the model that it will receive N candidates and must select a corresponding ID for the entry it 'thinks' corresponds best to the input query. We instruct the model to output an ID from 1 to N, the ID corresponding to the candidate entry number.

We also specify several guidelines to the model and then provide an example of how the results object will be presentred to the Generateive model, (we will write the code that formats the VectorStore results object to this XML format shortly.)

In [ ]:
def make_classification_system_prompt(n: int) -> str:
    CLASSIFICATION_SYSTEM_PROMPT = """You are an AI assistant designed to classify a user query based on the provided context. You will be provided with candidate entries retrieved from a knowledge base, each containing an ID and a text description. Your task is to analyze the user query and the text of the context entries to determine which of the entries best matches the user query.

         Guidelines:
         1. Always prioritize the provided context when making your classification.
         2. The context will be provided as an XML structure containing multiple entries. Each entry includes an ID and a text description.
         3. The IDs will be integer values from 1 to {n}, corresponding to the {n} candidate entries.
         4. Use the text of the entries to determine the most relevant classification for the user query.
         5. Your output must be a JSON object that adheres to the following schema:
         - The JSON object must contain a single key, `classification`.
         - The value of `classification` must be an integer between 1 and {n}, representing the ID of the best matching entry.
         - If no classification can be determined due to ambiguity or insufficient information, the value of `classification` must be `-1`.

         Example of the required JSON output:
         {{
              'classification': 1
         }}

         The XML structure for the context and user query will be as follows:
         <Context>
              <Entry>
                   <ID>0</ID>
                   <Text>[Text from the first entry]</Text>
              </Entry>
              <Entry>
                   <ID>1</ID>
                   <Text>[Text from the second entry]</Text>
              </Entry>
              ...
              <Entry>
                   <ID>{n}</ID>
                   <Text>[Text from the fifth entry]</Text>
              </Entry>
         </Context>

         <UserQuery>
              <Text>[The user query will be inserted here]</Text>
         </UserQuery>

         Your task is to analyze the context and the user query, and return the classification in the required structured format.
         """

    return CLASSIFICATION_SYSTEM_PROMPT.format(n=n)


system_prompt = make_classification_system_prompt(n=7)
print(system_prompt)

Next, we specify a Pydantic class model that will guide the structure of the output. Similar to our ealier example, we now want the generative model to output an ID between 1 and N, adhering the the System Prompt above.

In [ ]:
from typing import Literal

from pydantic import BaseModel, Field, conint, create_model


def make_classification_model(n: int) -> type[BaseModel]:
    if n < 1:
        raise ValueError("n must be >= 1")

    PositiveId = conint(ge=1, le=n)  # type: ignore[valid-type]

    return create_model(
        "ClassificationResponseModel",
        classification=(
            Literal[-1] | PositiveId,
            Field(description=f"-1 if unclassifiable, else an integer in [1, {n}] (no 0)."),
        ),
    )

In this case we have created a funcition that instructs the model to output either -1, or a value between 1 and N where N can be dynamically scaled depending on how many results are passed to the model.

### Pre-processing and post processing for the GenAI agent

We now need methods for:

- formatting the query and results object to string format to present to the agent
- formatting the response from the agent and selecting the final result based on the agent generation.

Our query-results formatting is straight forward - we want to convert from the VectorStoreSearchOutput object to the XML format specified earlier in the system prompt. Python code function to do this is as follows:

In [ ]:
import pandas as pd


def format_prompt_with_vectorstore_results(results_df) -> str:
    # Extract the user query (assuming all rows have the same query_id and query_text)
    user_query = results_df["query_text"].iloc[0]

    # Build the <Context> section
    context_entries = "\n".join(
        f"    <Entry>\n        <ID>{idx + 1}</ID>\n        <Text>{row['doc_text']}</Text>\n    </Entry>"
        for idx, row in results_df.iterrows()
    )

    # Combine everything into the final prompt
    formatted_prompt = f"""
<Context>
{context_entries}
</Context>

<UserQuery>
    <Text>{user_query}</Text>
</UserQuery>"""

    return formatted_prompt

Creating a dummy VectorStoreSearchOutput object we can see how the above function transforms the data that will be passed as the prompt to the agent model:

In [ ]:
from classifai.indexers.dataclasses import VectorStoreSearchOutput

# Create a sample dataframe adhering to the searchOutputSchema
sample_data = {
    "query_id": ["1", "1", "1", "1", "1"],
    "query_text": [
        "a photographer hired for wedding events",
        "a photographer hired for wedding events",
        "a photographer hired for wedding events",
        "a photographer hired for wedding events",
        "a photographer hired for wedding events",
    ],
    "doc_id": ["701", "702", "703", "704", "705"],
    "doc_text": [
        "Wedding photographer available for hire.",
        "Professional event photographer for weddings.",
        "Experienced celebration event photographer.",
        "Photographer specializing in nature photography.",
        "A fashion model photographer.",
    ],
    "rank": [0, 1, 2, 3, 4],
    "score": [0.95, 0.90, 0.85, 0.80, 0.75],
}

# Wrap the validated dataframe in the VectorStoreSearchOutput class
sample_results_object = VectorStoreSearchOutput(sample_data)


# pass the sample results object to the formatting function and print the output
print(format_prompt_with_vectorstore_results(sample_results_object))

print("\n\n-----\n\n")
# showing the original result object for compariosn with the XML string
sample_results_object

Focusing on the output of the model, we expect either a value of -1 or a value between 1 AND N. We write a function that validates that output, and then reduces the original VectorStoreSearchOutput object down to a single row result, based on the output label. If the agent returns -1 or an invalid response, we design this function to play it safe and return the original results object with no modifications. Although, the design choice here can be adapted for other functionalities - such as just returning the top ranked item with no agent assessment or otherwise.

In [ ]:
import json


def format_agent_classification(
    agent_generated_text: str, results_df: VectorStoreSearchOutput
) -> VectorStoreSearchOutput:
    # Parse the generated text
    try:
        response = json.loads(agent_generated_text)
        validation_model = make_classification_model(n=results_df.shape[0])
        validated_response = validation_model(**response)
    except (json.JSONDecodeError, ValueError):
        # If parsing or validation fails, return the original DataFrame
        return results_df

    # Extract the classification
    classification = validated_response.classification

    # Validate the classification value is in the expected range
    MIN_INDEX = 1
    MAX_INDEX = len(results_df)
    if int(classification) < MIN_INDEX or int(classification) > MAX_INDEX:
        return results_df

    # Otherwise, filter to only keep the row with the classified doc_id, adjusting for 1-based 0-indexing in the classification
    result = results_df.iloc[[classification - 1]].reset_index(drop=True)

    return VectorStoreSearchOutput(result)

We can try out this function as well using our example result_df and for now a manually generated agent json response.

In [ ]:
genai_response = {"classification": "3"}

classified_result_df = format_agent_classification(json.dumps(genai_response), sample_results_object)

classified_result_df

If you change the classification label value, you'll see different results being selected, and if the classificaiton label is set to -1, 0 or greater than 5, or some other invalid value it will return the full dataset.

### Instantiating the Agent

We now want to pull all this together and demonstrate the generative agent performing the above task.

In [ ]:
from google import genai
from google.genai import types

# Set up the API key for authentication
client = genai.Client(api_key="YOUR API KEY HERRE")

# Define the model to use
chosen_model = "gemini-2.5-flash"  # Replace with the appropriate Gemini Flash 2.5 model name if different


prompt5 = format_prompt_with_vectorstore_results(sample_results_object)


response = client.models.generate_content(
    model=chosen_model,
    contents=prompt5,
    config=types.GenerateContentConfig(
        system_instruction=make_classification_system_prompt(n=len(sample_results_object)),
        response_mime_type="application/json",
        response_json_schema=make_classification_model(n=len(sample_results_object)).model_json_schema(),
    ),
)


classification_df = format_agent_classification(response.text, sample_results_object)

classification_df

Done! The model does take a few seconds to consider, but here we have a VectorStore result, being formatted into a prompt, the model is provided a system instruction and structured Pydantic guidance on what to produce with the provided information, and a post-processsing function that outputs the answer

### Building a Hook to process results with the agent

All thats left to do is to code the post-processing search() method hook, which we be called to operate on the VectorStoreSearchOutput object after the method returns that object. We provide the full hook implementation in the next with extra comments, typing together the various components we've discussed up until now

In [ ]:
# reinstiating the genai Client outside of the function to avoid re-instantiating on every call
client = genai.Client(api_key="YOUR API KEY HERE")
chosen_model = "gemini-2.5-flash"  # we could choose another model, for example gemini-3 vraiants are available

# we've alreadt defined the CLASSIFICATION_SYSTEM_PROMPT,  format_prompt_with_vectorstore_results and the make_classification_model function, so we recan use those directly without redefining


def rag_classifier(input_data: VectorStoreSearchOutput) -> VectorStoreSearchOutput:
    # first our result df might have multiple queries in it, so we want to group by query_id and query_text to ensure we classify each unique query separately
    per_query = list(input_data.groupby(["query_id"]))

    # creating an empty list to store the classified results for each query
    classified_results = []

    # iterate over the object for each query
    for _, group in per_query:
        # first we want to set a manual maximum limit on how many samples the model can classify, as the more samples we include the more context we provide but also the more expensive the model call will be. Here we take the top 5 results for each query, but this could be adjusted based on the expected number of results and the cost/benefit tradeoff of providing more context to the model
        # and code can handle any value of N, this is a practical limit relating to the generateive models context window and effectiveness.
        group = group.head(5)  # noqa: PLW2901

        # pass the group dataframe to the formatting function to get the prompt for this query
        prompt = format_prompt_with_vectorstore_results(group)

        # send the prompt to the model and get the response, passing the same system instruction and structured json schema as before
        response = client.models.generate_content(
            model=chosen_model,
            contents=prompt,
            config=types.GenerateContentConfig(
                # there are many other config options we could experiment with here, for example temperature, nucleas sampling, top_k, top_p, and max_response_tokens
                system_instruction=make_classification_system_prompt(n=group.shape[0]),
                response_mime_type="application/json",
                response_json_schema=make_classification_model(n=group.shape[0]).model_json_schema(),
            ),
        )

        # we try to pass the result to our postprocessing function which will either return the chosen classificaiton, or on failure return the original result
        classified_group = format_agent_classification(response.text, group)

        # the classification for that query is added to the classified_results list
        classified_results.append(classified_group)

    # we convert the results to a single dataframe
    final_result = pd.concat(classified_results).reset_index(drop=True)

    # we wrap the final result in the VectorStoreSearchOutput class to ensure it adheres to the expected schema
    hook_output = VectorStoreSearchOutput(final_result)

    return hook_output

Lets see how that works when we apply it to a VectorStore as a Hook!

In [ ]:
from classifai.indexers import VectorStore
from classifai.vectorisers import GcpVectoriser

# Initialise the vectoriser
demo_vectoriser = GcpVectoriser(project_id="YOUR PROJECT ID HERE", vertexai=True)

# Initialise the vector store, pointing the demo to the demo test data
demo_vectorstore = VectorStore(
    file_name="./data/fake_soc_dataset.csv",
    data_type="csv",
    vectoriser=demo_vectoriser,
    output_dir="./demo_vdb",
    overwrite=True,
    hooks={"search_postprocess": rag_classifier},
)

In [ ]:
from classifai.indexers import VectorStoreSearchInput

search_input = VectorStoreSearchInput(
    {"id": ["1", "2"], "query": ["a photographer hired for wedding events", "Tax expert advisor"]}
)

result = demo_vectorstore.search(search_input, n_results=5)

result

Thats it! Without the added generative hook we would see a set of candidate results in each case here. Dynamically removing the genai hook we can see what results were considered by the Generative RAG model:

In [ ]:
demo_vectorstore.hooks = {}
result_no_genai = demo_vectorstore.search(search_input, n_results=5)
result_no_genai

## That's it! We've walked through:

- Establishing connections to Google Generative AI models,

- Configuring AI model input and output for Classification tasks and how models should behave,

- Setting up pre- and post-processing functions to handle the model input and output.

In this last part of this notebook (below) we provide a full code implementation for a VectorStore, and GenAI classifier hook, but here are some additional final points to consider:

- For each query and result, we call the Google API sequentially - which is fine but can be slow - in the below final imp we show how it can be done in an asynchronous manner for each query to speed things up.

- We didn't add any additional checks to make sure the user query prompt would not break the pre-processing formatting function , such as the user prompt containing punctuation that may break the formatter.

- There are many different models available to consider with different trade offs in cost and capability as well as many metadata arguments that can affect performance, while our guide shows good practices using systems prompts and other features it is worth taking time to reviewing the Google documentation on how best to use these models and how to acheive constistent performance.

## Typing it all together

The following code cells provide a complete implementation of the Generative AI RAG agent,  (reimporting dependecies and reinitialising all variables)

In this version we provide one key change with an _asyncronous_ version of the API calls to the genai client, to speed up the processing of queries. Note that ClassifAI does not currently support async code in hooks, therefore we nest our agent code in an outer syncronous function.

`Warning:` To run this final section of the code you will need to execute it outside of the Notebook environments, as it contains async code that does not operate nicely within the Jupyter Notebook environment. We recommend exceuting in an external script with ClassifAI[gcp] installed in the environment.

In [ ]:
import asyncio
import json

from google import genai
from google.genai import types
from pydantic import BaseModel, Field

from classifai.indexers.dataclasses import VectorStoreSearchInput, VectorStoreSearchOutput


# thia function generates the system prompt for the classification task, it takes the number of candidate entries as input and returns a formatted string that includes instructions for the model on how to classify the user query based on the provided context. The prompt specifies the expected format of the input and output, and provides guidelines for how the model should prioritize the context when making its classification
def make_classification_system_prompt(n: int) -> str:
    CLASSIFICATION_SYSTEM_PROMPT = """You are an AI assistant designed to classify a user query based on the provided context. You will be provided with candidate entries retrieved from a knowledge base, each containing an ID and a text description. Your task is to analyze the user query and the text of the context entries to determine which of the entries best matches the user query.

         Guidelines:
         1. Always prioritize the provided context when making your classification.
         2. The context will be provided as an XML structure containing multiple entries. Each entry includes an ID and a text description.
         3. The IDs will be integer values from 1 to {n}, corresponding to the {n} candidate entries.
         4. Use the text of the entries to determine the most relevant classification for the user query.
         5. Your output must be a JSON object that adheres to the following schema:
         - The JSON object must contain a single key, `classification`.
         - The value of `classification` must be an integer between 1 and {n}, representing the ID of the best matching entry.
         - If no classification can be determined due to ambiguity or insufficient information, the value of `classification` must be `-1`.

         Example of the required JSON output:
         {{
              'classification': 1
         }}

         The XML structure for the context and user query will be as follows:
         <Context>
              <Entry>
                   <ID>0</ID>
                   <Text>[Text from the first entry]</Text>
              </Entry>
              <Entry>
                   <ID>1</ID>
                   <Text>[Text from the second entry]</Text>
              </Entry>
              ...
              <Entry>
                   <ID>{n}</ID>
                   <Text>[Text from the fifth entry]</Text>
              </Entry>
         </Context>

         <UserQuery>
              <Text>[The user query will be inserted here]</Text>
         </UserQuery>

         Your task is to analyze the context and the user query, and return the classification in the required structured format.
         """

    return CLASSIFICATION_SYSTEM_PROMPT.format(n=n)


# this function generates a Pydantic model for validating the model's response, it takes the number of candidate entries as input and returns a dynamically created Pydantic model class that includes a single field 'classification' which can be either -1 or an integer between 1 and n. This model is used to ensure that the response from the generative model adheres to the expected format and value constraints
def make_classification_model(n: int) -> type[BaseModel]:
    if n < 1:
        raise ValueError("n must be >= 1")

    PositiveId = conint(ge=1, le=n)  # type: ignore[valid-type]

    return create_model(
        "ClassificationResponseModel",
        classification=(
            Literal[-1] | PositiveId,
            Field(description=f"-1 if unclassifiable, else an integer in [1, {n}] (no 0)."),
        ),
    )


# this function convertes the VectorStoreSearchOutput to a formatted string that can be passed as a prompt to the generative model, it extracts the user query and the candidate entries and formats them into the required XML structure as specified in the system prompt
def format_prompt_with_vectorstore_results(results_df) -> str:
    # Extract the user query (assuming all rows have the same query_id and query_text)
    user_query = results_df["query_text"].iloc[0]

    # Build the <Context> section
    context_entries = "\n".join(
        f"    <Entry>\n        <ID>{idx + 1}</ID>\n        <Text>{row['doc_text']}</Text>\n    </Entry>"
        for idx, row in results_df.iterrows()
    )

    # Combine everything into the final prompt
    formatted_prompt = f"""
<Context>
{context_entries}
</Context>

<UserQuery>
    <Text>{user_query}</Text>
</UserQuery>"""

    return formatted_prompt


# this function takes the generative model's response and the original results dataframe, it attempts to parse and validate the model's response, and if successful it filters the original results dataframe to return only the row corresponding to the classification returned by the model. If parsing or validation fails, or if the classification is out of range, it returns the original results dataframe unfiltered
def format_agent_classification(
    agent_generated_text: str, results_df: VectorStoreSearchOutput
) -> VectorStoreSearchOutput:
    # Parse the generated text
    try:
        response = json.loads(agent_generated_text)
        validation_model = make_classification_model(n=results_df.shape[0])
        validated_response = validation_model(**response)
    except (json.JSONDecodeError, ValueError):
        # If parsing or validation fails, return the original DataFrame
        return results_df

    # Extract the classification
    classification = validated_response.classification

    # Validate the classification value is in the expected range
    MIN_INDEX = 1
    MAX_INDEX = len(results_df)
    if int(classification) < MIN_INDEX or int(classification) > MAX_INDEX:
        return results_df

    # Otherwise, filter to only keep the row with the classified doc_id, adjusting for 1-based 0-indexing in the classification
    result = results_df.iloc[[classification - 1]].reset_index(drop=True)

    return VectorStoreSearchOutput(result)


# herre we restart a client instance to ensure we have access to the async interface, as per the SDK documentation you shared, we should create a new client instance for the async code and then close it after use to avoid any potential issues with reusing the same client instance across sync and async code
client = genai.Client(api_key="YOUR API KEY HERE")
chosen_model = "gemini-2.5-flash"  # could use another model here if desired, for example a gemini-3 variant
MAX_CONCURRENCY = 8


# this function is an async version of the classification function for a single query group, it takes the async client, the results dataframe for a single query, and a semaphore to limit concurrency. It formats the prompt, sends the request to the model, and returns the classified result
async def _classify_result(
    aclient,
    result: VectorStoreSearchOutput,
    sem: asyncio.Semaphore,
) -> VectorStoreSearchOutput:
    group = result.head(5)
    prompt = format_prompt_with_vectorstore_results(group)
    n = group.shape[0]

    async with sem:
        response = await aclient.models.generate_content(
            model=chosen_model,
            contents=prompt,
            config=types.GenerateContentConfig(
                system_instruction=make_classification_system_prompt(n=n),
                response_mime_type="application/json",
                response_json_schema=make_classification_model(n=n).model_json_schema(),
            ),
        )

    return format_agent_classification(response.text, group)


# this funcion is the main async function that orchestrates the classification of the results, it groups the results by query, creates async tasks for each group, and gathers the results. Finally, it concatenates the classified results and returns them wrapped in the VectorStoreSearchOutput class
async def rag_classifier_async(
    input_data: VectorStoreSearchOutput,
) -> VectorStoreSearchOutput:
    # first our result df might have multiple queries in it, so we want to group by query_id and query_text to ensure we classify each unique query separately
    per_query = [group for _, group in input_data.groupby(["query_id"])]

    # This is the SDK-provided async interface
    aclient = client.aio
    sem = asyncio.Semaphore(MAX_CONCURRENCY)

    try:
        tasks = [_classify_result(aclient, group, sem) for group in per_query]
        classified_results = await asyncio.gather(*tasks)
    finally:
        # Important per the SDK docs/snippet you pasted
        await aclient.aclose()

    # we convert the results to a single dataframe
    final_result = pd.concat(classified_results).reset_index(drop=True)
    return VectorStoreSearchOutput(final_result)


# this function is a simple wrapper around the async classification function to allow it to be used as a hook in the VectorStore, it runs the async function using asyncio.run and returns the result
def rag_hook(input_data: VectorStoreSearchOutput) -> VectorStoreSearchOutput:
    return asyncio.run(rag_classifier_async(input_data))

In [ ]:
from classifai.indexers import VectorStore
from classifai.vectorisers import GcpVectoriser

# We instantiate a vectoriser (we could use any Vectoriser from ClassifAI here, but we'll use the GCP one to keep it consistent with the rest of the demo)
imp_vectoriser = GcpVectoriser(project_id="YOUR GCP PROJECT ID HERE", vertexai=True)


# Build the vectorstore which points to the demo data and includes the RAG classification hook in the search_postprocess step, this means that every time we call search on this vectorstore, after it retrieves the candidate entries it will pass them to the rag_hook function which will classify the results using the Gemini model before returning them
imp_vectorstore = VectorStore(
    file_name="./data/fake_soc_dataset.csv",
    data_type="csv",
    vectoriser=imp_vectoriser,
    output_dir="./demo_vdb",
    overwrite=True,
    hooks={"search_postprocess": rag_hook},
)

In [ ]:
# create a new demo input dataframe to showcase the RAG classification being performed asynchronously.
imp_search_input = VectorStoreSearchInput(
    {
        "id": ["1", "2", "3", "4", "5"],
        "query": [
            "a photographer hired for wedding events",
            "Tax expert advisor",
            "farmer of potatoes",
            "Tax expert advisor",
            "farmer of potatoes",
        ],
    }
)

# finally run the VectorStore search method which will call the RAG classification hook we have defined on the search results
imp_vectorstore.search(imp_search_input, n_results=5)